# CRISP Predictive Modeling Pipeline - End-to-End Demonstration

**Complete walkthrough of the predictive modeling pipeline from data to deployment**

This notebook provides a comprehensive, interactive demonstration of the entire CRISP predictive modeling process.

## **What This Notebook Does**
1. **Feature Extraction** - Extract time series and static features from OMOP data
2. **Traditional Model Training** - Train LogisticRegression, RandomForest, GradientBoosting, XGBoost
3. **Deep Learning Training** - Train MLP (Multi-Layer Perceptron) models
4. **Model Evaluation** - Generate performance metrics and visualizations
5. **Results Analysis** - Compare models and provide recommendations

## **Clinical Prediction Tasks**
- **Mortality**: 7-day, 30-day ICU mortality
- **Readmission**: 7-day, 30-day, 90-day readmission
- **Length of Stay**: >3 days, >7 days
- **Sepsis**: 24h, 48h, during ICU sepsis

In [ ]:
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import warnings
import time
from datetime import datetime
import os

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("CRISP Predictive Modeling Pipeline - Interactive Demo")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)
print("\nThis notebook will execute the complete pipeline step by step")
print("   You can follow along and see the results in real-time!")

## Step 1: Data Preparation Check

First, let's verify that we have the required patient data for feature extraction.

In [ ]:
def check_data_availability():
    """Check if patient data is available for feature extraction"""
    print("**CHECKING DATA AVAILABILITY**\n")
    
    # Check for patient data
    data_dir = Path('../../extracted_patient_data')
    if data_dir.exists():
        patient_files = list(data_dir.rglob('patient_labels.json'))
        print(f"[SUCCESS] Found patient data directory: {data_dir}")
        print(f"Number of patients with labels: {len(patient_files)}")
        
        if patient_files:
            # Sample one patient to show structure
            sample_file = patient_files[0]
            with open(sample_file, 'r') as f:
                sample_data = json.load(f)
            
            print(f"\n**Sample Patient Data Structure:**")
            print(f"   Patient ID: {sample_data.get('patient_id', 'N/A')}")
            print(f"   ICU Admission: {sample_data.get('has_icu_admission', 'N/A')}")
            
            # Show available prediction targets
            targets = []
            if 'mortality' in sample_data:
                targets.extend([f"Mortality: {list(sample_data['mortality'].keys())}"]) 
            if 'readmission' in sample_data:
                targets.extend([f"Readmission: {list(sample_data['readmission'].keys())}"])
            if 'los' in sample_data:
                targets.extend([f"Length of Stay: {list(sample_data['los'].keys())}"])
            if any(k.startswith('sepsis') or k.startswith('has_sepsis') for k in sample_data.keys()):
                sepsis_keys = [k for k in sample_data.keys() if 'sepsis' in k]
                targets.extend([f"Sepsis: {sepsis_keys}"])
            
            print(f"\n**Available Prediction Targets:**")
            for target in targets:
                print(f"   - {target}")
            
            return True, len(patient_files)
        else:
            print("[FAILED] No patient label files found")
            return False, 0
    else:
        print(f"[FAILED] Patient data directory not found: {data_dir}")
        print("\n**To generate patient data:**")
        print("   1. Run the CRISP pipeline modules 1-5 first")
        print("   2. Ensure extracted_patient_data/ directory exists")
        return False, 0

# Check data availability
data_available, num_patients = check_data_availability()

if data_available:
    print(f"\n**Ready to proceed!** Found data for {num_patients} patients")
else:
    print(f"\n**Cannot proceed without patient data**")
    print("   Please run the CRISP data extraction pipeline first")

## Step 2: Feature Extraction

Extract time series features (for mortality/readmission/LOS) and static features (for sepsis prediction).

In [ ]:
def run_feature_extraction():
    """Execute feature extraction step"""
    print("**STEP 1: FEATURE EXTRACTION**\n")
    print("Extracting features from patient data...")
    print("- Time series features: 4-hour windows, 24-hour observation")
    print("- Static features: Demographics + pre-ICU conditions")
    
    start_time = time.time()
    
    # Run feature extraction
    result = subprocess.run([
        'python', '1_feature_engineering/run_feature_extraction.py',
        '--time-window', '4',
        '--min-observation', '24',
        '--output-dir', 'modeling_results/features'
    ], 
    cwd='../', 
    capture_output=True, 
    text=True)
    
    duration = time.time() - start_time
    
    if result.returncode == 0:
        print(f"\n[SUCCESS] Feature extraction completed in {duration:.1f}s")
        
        # Parse output for key statistics
        lines = result.stdout.split('\n')
        for line in lines:
            if 'Samples:' in line or 'Features:' in line or 'Data completeness:' in line:
                print(f"   {line.strip()}")
        
        # Check generated files
        features_dir = Path('../modeling_results/features')
        if features_dir.exists():
            feature_files = list(features_dir.glob('*.csv'))
            print(f"\nGenerated {len(feature_files)} feature files:")
            for f in feature_files:
                print(f"   {f.name}")
        
        return True
    else:
        print(f"\n[FAILED] Feature extraction failed after {duration:.1f}s")
        print(f"Error: {result.stderr}")
        return False

# Run feature extraction if data is available
if data_available:
    feature_success = run_feature_extraction()
else:
    print("[SKIP] Skipping feature extraction - no patient data available")
    feature_success = False

## Step 3: Traditional Model Training

Train traditional machine learning models (LogisticRegression, RandomForest, GradientBoosting, XGBoost).

In [ ]:
def run_traditional_training():
    """Execute traditional model training"""
    print("**STEP 2: TRADITIONAL MODEL TRAINING**\n")
    print("Training models: LogisticRegression, RandomForest, GradientBoosting, XGBoost")
    print("Tasks: Mortality, Readmission, Length of Stay, Sepsis")
    
    start_time = time.time()
    
    # Run traditional model training
    result = subprocess.run([
        'python', '2_model_training/run_traditional_models.py',
        '--input-dir', 'modeling_results/features',
        '--output-dir', 'modeling_results/models/traditional',
        '--time-window', '4',
        '--tasks', 'mortality,readmission,los,sepsis',
        '--models', 'LogisticRegression,RandomForest,GradientBoosting,XGBoost'
    ], 
    cwd='../', 
    capture_output=True, 
    text=True)
    
    duration = time.time() - start_time
    
    if result.returncode == 0:
        print(f"\n[SUCCESS] Traditional model training completed in {duration:.1f}s")
        
        # Parse output for performance summary
        lines = result.stdout.split('\n')
        in_summary = False
        for line in lines:
            if 'TRAINING SUMMARY' in line:
                in_summary = True
            elif in_summary and ('MORTALITY:' in line or 'READMISSION:' in line or 'LOS:' in line or 'SEPSIS:' in line):
                print(f"   {line.strip()}")
            elif in_summary and ('Targets trained:' in line or 'Average AUROC:' in line or 'Best:' in line):
                print(f"      {line.strip()}")
        
        # Check generated models
        models_dir = Path('../modeling_results/models/traditional')
        if models_dir.exists():
            model_files = list(models_dir.rglob('*.pkl'))
            print(f"\nTrained {len(model_files)} traditional models")
        
        return True
    else:
        print(f"\n[FAILED] Traditional model training failed after {duration:.1f}s")
        print(f"Error: {result.stderr}")
        return False

# Run traditional model training if features are available
if 'feature_success' in locals() and feature_success:
    traditional_success = run_traditional_training()
else:
    print("[SKIP] Skipping traditional model training - features not available")
    traditional_success = False

## Step 4: Deep Learning Model Training

Train deep learning models (MLP - Multi-Layer Perceptron).

In [ ]:
def run_deep_learning_training():
    """Execute deep learning model training"""
    print("**STEP 3: DEEP LEARNING MODEL TRAINING**\n")
    print("Training MLP (Multi-Layer Perceptron) models")
    print("Architecture: Input -> Dense(256) -> Dense(128) -> Dense(64) -> Output")
    
    start_time = time.time()
    
    # Run deep learning model training
    result = subprocess.run([
        'python', '2_model_training/run_DL_models.py',
        '--input-dir', 'modeling_results/features',
        '--output-dir', 'modeling_results/models/deep_learning',
        '--time-window', '4',
        '--model-type', 'MLP',
        '--tasks', 'mortality,readmission,los,sepsis',
        '--epochs', '100',
        '--batch-size', '32'
    ], 
    cwd='../', 
    capture_output=True, 
    text=True)
    
    duration = time.time() - start_time
    
    if result.returncode == 0:
        print(f"\n[SUCCESS] Deep learning training completed in {duration:.1f}s")
        
        # Parse output for performance summary
        lines = result.stdout.split('\n')
        in_summary = False
        for line in lines:
            if 'TRAINING SUMMARY' in line:
                in_summary = True
            elif in_summary and ('MORTALITY:' in line or 'READMISSION:' in line or 'LOS:' in line or 'SEPSIS:' in line):
                print(f"   {line.strip()}")
            elif in_summary and ('Targets trained:' in line or 'Average AUROC:' in line or 'Best:' in line):
                print(f"      {line.strip()}")
        
        # Check generated models
        models_dir = Path('../modeling_results/models/deep_learning')
        if models_dir.exists():
            model_files = list(models_dir.rglob('*.pt'))
            print(f"\nTrained {len(model_files)} deep learning models")
        
        return True
    else:
        print(f"\n[FAILED] Deep learning training failed after {duration:.1f}s")
        print(f"Error: {result.stderr}")
        return False

# Run deep learning training if traditional models succeeded
if 'traditional_success' in locals() and traditional_success:
    dl_success = run_deep_learning_training()
else:
    print("[SKIP] Skipping deep learning training - traditional models not available")
    dl_success = False

## Step 5: Model Evaluation

Evaluate all trained models and generate comprehensive performance reports.

In [ ]:
def run_model_evaluation():
    """Execute model evaluation"""
    print("**STEP 4: MODEL EVALUATION**\n")
    print("Evaluating all trained models...")
    print("Generating: AUROC/AUPRC metrics, ROC curves, PR curves, reports")
    
    start_time = time.time()
    
    # Run model evaluation
    result = subprocess.run([
        'python', '3_evaluation/run_evaluation.py',
        '--model-dir', 'modeling_results/models',
        '--feature-dir', 'modeling_results/features',
        '--output-dir', 'modeling_results/evaluation',
        '--time-window', '4'
    ], 
    cwd='../', 
    capture_output=True, 
    text=True)
    
    duration = time.time() - start_time
    
    if result.returncode == 0:
        print(f"\n[SUCCESS] Model evaluation completed in {duration:.1f}s")
        
        # Check generated files
        eval_dir = Path('../modeling_results/evaluation')
        if eval_dir.exists():
            plot_files = list(eval_dir.rglob('*.png'))
            json_files = list(eval_dir.rglob('*.json'))
            report_files = list(eval_dir.rglob('*.md'))
            
            print(f"\nGenerated evaluation outputs:")
            print(f"   Plots: {len(plot_files)}")
            print(f"   Reports: {len(report_files)}")
            print(f"   Metrics: {len(json_files)}")
        
        return True
    else:
        print(f"\n[FAILED] Model evaluation failed after {duration:.1f}s")
        print(f"Error: {result.stderr}")
        return False

# Run evaluation if models are available
if ('traditional_success' in locals() and traditional_success) or ('dl_success' in locals() and dl_success):
    eval_success = run_model_evaluation()
else:
    print("[SKIP] Skipping model evaluation - no trained models available")
    eval_success = False

## Step 6: Results Analysis and Comparison

Load and analyze all results to provide model performance comparison and recommendations.

In [ ]:
def load_and_analyze_results():
    """Load all results and create comprehensive analysis"""
    print("**STEP 5: RESULTS ANALYSIS**\n")
    
    results_data = []
    models_found = 0
    
    # Load traditional model results
    trad_dir = Path('../modeling_results/models/traditional')
    if trad_dir.exists():
        for model_dir in trad_dir.iterdir():
            if model_dir.is_dir():
                model_name = model_dir.name.replace('_', ' ').title()
                for metrics_file in model_dir.glob('*_metrics.json'):
                    parts = metrics_file.stem.replace('_metrics', '').split('_')
                    if len(parts) >= 2:
                        task = parts[0]
                        target = '_'.join(parts[1:])
                        
                        with open(metrics_file, 'r') as f:
                            metrics = json.load(f)
                        
                        results_data.append({
                            'Model_Type': 'Traditional',
                            'Model_Name': model_name,
                            'Task': task.title(),
                            'Target': target.replace('_', ' ').title(),
                            'AUROC': metrics.get('auroc', 0),
                            'AUPRC': metrics.get('auprc', 0)
                        })
                        models_found += 1
    
    # Load deep learning results
    dl_dir = Path('../modeling_results/models/deep_learning')
    if dl_dir.exists():
        for model_dir in dl_dir.iterdir():
            if model_dir.is_dir():
                model_name = model_dir.name.upper()
                for metrics_file in model_dir.glob('*_metrics.json'):
                    parts = metrics_file.stem.replace('_metrics', '').split('_')
                    if len(parts) >= 2:
                        task = parts[0]
                        target = '_'.join(parts[1:])
                        
                        with open(metrics_file, 'r') as f:
                            metrics = json.load(f)
                        
                        results_data.append({
                            'Model_Type': 'Deep Learning',
                            'Model_Name': model_name,
                            'Task': task.title(),
                            'Target': target.replace('_', ' ').title(),
                            'AUROC': metrics.get('auroc', 0),
                            'AUPRC': metrics.get('auprc', 0)
                        })
                        models_found += 1
    
    if results_data:
        df = pd.DataFrame(results_data)
        df['Combined_Score'] = 0.6 * df['AUROC'] + 0.4 * df['AUPRC']
        
        print(f"**PERFORMANCE SUMMARY**")
        print(f"   Total model evaluations: {len(df)}")
        print(f"   Model types: {', '.join(df['Model_Type'].unique())}")
        print(f"   Tasks: {', '.join(df['Task'].unique())}")
        print(f"   Best AUROC: {df['AUROC'].max():.3f}")
        print(f"   Best AUPRC: {df['AUPRC'].max():.3f}")
        
        return df
    else:
        print("[FAILED] No model results found to analyze")
        return None

# Load and analyze results if evaluation succeeded
if 'eval_success' in locals() and eval_success:
    results_df = load_and_analyze_results()
else:
    print("[SKIP] Skipping results analysis - no evaluation results available")
    results_df = None

## Step 7: Performance Visualization

Create visualizations to compare model performance across tasks.

In [ ]:
if results_df is not None and len(results_df) > 0:
    print("**PERFORMANCE VISUALIZATION**\n")
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. AUROC comparison by model type
    sns.boxplot(data=results_df, x='Model_Type', y='AUROC', ax=axes[0,0])
    axes[0,0].set_title('AUROC Distribution by Model Type', fontsize=14)
    axes[0,0].set_ylim(0.5, 1.0)
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. AUPRC comparison by model type
    sns.boxplot(data=results_df, x='Model_Type', y='AUPRC', ax=axes[0,1])
    axes[0,1].set_title('AUPRC Distribution by Model Type', fontsize=14)
    axes[0,1].set_ylim(0, 1.0)
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Performance by task
    task_performance = results_df.groupby(['Task', 'Model_Type'])['AUROC'].mean().reset_index()
    sns.barplot(data=task_performance, x='Task', y='AUROC', hue='Model_Type', ax=axes[1,0])
    axes[1,0].set_title('Average AUROC by Task', fontsize=14)
    axes[1,0].tick_params(axis='x', rotation=45)
    axes[1,0].set_ylim(0.5, 1.0)
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. AUROC vs AUPRC scatter
    colors = {'Traditional': 'blue', 'Deep Learning': 'red'}
    for model_type in results_df['Model_Type'].unique():
        data = results_df[results_df['Model_Type'] == model_type]
        axes[1,1].scatter(data['AUROC'], data['AUPRC'], 
                         c=colors.get(model_type, 'gray'), 
                         label=model_type, alpha=0.7, s=80)
    
    axes[1,1].set_xlabel('AUROC')
    axes[1,1].set_ylabel('AUPRC')
    axes[1,1].set_title('AUROC vs AUPRC Performance', fontsize=14)
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Display top performing models
    print("\n**TOP PERFORMING MODELS**\n")
    top_models = results_df.nlargest(5, 'Combined_Score')[['Model_Name', 'Model_Type', 'Task', 'Target', 'AUROC', 'AUPRC', 'Combined_Score']]
    display(top_models)
    
else:
    print("No results available for visualization")

## Step 8: Pipeline Summary

Comprehensive summary of the entire pipeline execution.

In [ ]:
def display_pipeline_summary():
    """Display comprehensive pipeline summary"""
    print("**PIPELINE EXECUTION SUMMARY**\n")
    
    # Check pipeline completion status by examining actual files/directories
    steps_completed = []
    
    # Check for patient data
    data_dir = Path('../../extracted_patient_data')
    data_available = data_dir.exists() and list(data_dir.rglob('patient_labels.json'))
    
    if data_available:
        steps_completed.append("[SUCCESS] Data Check: Patient data found")
    else:
        steps_completed.append("[FAILED] Data Check: No patient data")
    
    # Check for feature files
    features_dir = Path('../modeling_results/features')
    feature_files = list(features_dir.glob('*.csv')) if features_dir.exists() else []
    feature_success = len(feature_files) > 0
    
    if feature_success:
        steps_completed.append("[SUCCESS] Feature Extraction: Completed successfully")
    else:
        steps_completed.append("[FAILED] Feature Extraction: Failed or skipped")
    
    # Check for traditional model files
    trad_models_dir = Path('../modeling_results/models/traditional')
    trad_model_files = list(trad_models_dir.rglob('*.pkl')) if trad_models_dir.exists() else []
    traditional_success = len(trad_model_files) > 0
    
    if traditional_success:
        steps_completed.append("[SUCCESS] Traditional Models: Trained successfully")
    else:
        steps_completed.append("[FAILED] Traditional Models: Failed or skipped")
    
    # Check for deep learning model files
    dl_models_dir = Path('../modeling_results/models/deep_learning')
    dl_model_files = list(dl_models_dir.rglob('*.pt')) if dl_models_dir.exists() else []
    dl_success = len(dl_model_files) > 0
    
    if dl_success:
        steps_completed.append("[SUCCESS] Deep Learning: Trained successfully")
    else:
        steps_completed.append("[FAILED] Deep Learning: Failed or skipped")
    
    # Check for evaluation files
    eval_dir = Path('../modeling_results/evaluation')
    eval_files = list(eval_dir.rglob('*.json')) if eval_dir.exists() else []
    eval_success = len(eval_files) > 0
    
    if eval_success:
        steps_completed.append("[SUCCESS] Evaluation: Completed successfully")
    else:
        steps_completed.append("[FAILED] Evaluation: Failed or skipped")
    
    print("**Pipeline Steps:**")
    for step in steps_completed:
        print(f"   {step}")
    
    # Check generated files
    results_dir = Path('../modeling_results')
    if results_dir.exists():
        feature_file_count = len(feature_files)
        model_file_count = len(trad_model_files) + len(dl_model_files)
        plot_files = list(results_dir.rglob('*.png'))
        plot_file_count = len(plot_files)
        
        print(f"\n**Generated Files:**")
        print(f"   Feature files: {feature_file_count}")
        print(f"   Model files: {model_file_count}")
        print(f"   Plot files: {plot_file_count}")
        print(f"\nAll results saved in: ../modeling_results/")
    else:
        print(f"\n**Generated Files:** No results directory found")
    
    # Final status
    successful_steps = sum(1 for step in steps_completed if '[SUCCESS]' in step)
    total_steps = len(steps_completed)
    
    print(f"\n**Final Status**: {successful_steps}/{total_steps} steps completed")
    
    if successful_steps == total_steps:
        print("**SUCCESS**: Complete pipeline executed successfully!")
        print("   Ready for model deployment and clinical validation")
    elif successful_steps > 0:
        print("**PARTIAL**: Some steps completed successfully")
        print("   Pipeline has generated results - check individual step outputs above")
    else:
        print("**FAILED**: No pipeline steps completed successfully")
        print("   Review error messages above and ensure data availability")

# Display final summary
display_pipeline_summary()

print(f"\n\n**Pipeline Demo Completed**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("**CRISP Predictive Modeling Pipeline - End-to-End Demonstration**")